# ISIC 2024 - Data Preprocessing Pipeline

This notebook prepares the metadata for model training.

The preprocessing pipeline is designed for:
- Tabular deep learning models
- Multimodal learning
- Future inference consistency

---

## Objectives

- Clean and standardize metadata
- Remove potential leakage features
- Handle missing values
- Encode categorical variables
- Normalize numerical features
- Generate image paths for multimodal training
- Save reusable preprocessing artifacts

---

## Important Considerations

- Preprocessing must be fit only on training data
- Validation and holdout sets must remain unseen
- Feature leakage must be avoided
- Consistent preprocessing is required for inference

In [16]:
import pandas as pd
import numpy as np
import random

RANDOM_STATE = 42

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

### Load Split Datasets

Previously generated patient-level splits are loaded for preprocessing.

The datasets include:
- Training set
- Validation set
- Holdout set

In [17]:
train_df = pd.read_csv("../data/splits/train_split.csv", low_memory=False)
val_df = pd.read_csv("../data/splits/val_split.csv", low_memory=False)
holdout_df = pd.read_csv("../data/splits/holdout_split.csv", low_memory=False)

print("Train:", train_df.shape)
print("Val:", val_df.shape)
print("Holdout:", holdout_df.shape)

Train: (320848, 55)
Val: (38002, 55)
Holdout: (42209, 55)


In [18]:
train_ids = train_df["isic_id"].copy()
val_ids = val_df["isic_id"].copy()
holdout_ids = holdout_df["isic_id"].copy()

### Feature Selection (Initial)

Removed:
- Diagnostic columns (`iddx_*`, `mel_*`) -> high missing / leakage risk
- Model-derived feature (`tbp_lv_dnn_lesion_confidence`) -> leakage

Only features available at inference time are kept.

### Remove Leakage and Identifier Columns

Some columns are removed because:
- They are unavailable during inference
- They directly leak diagnostic information
- They are patient or lesion identifiers
- They are not useful for model learning

In [19]:
DROP_COLUMNS = [
    # target leakage / not in test
    "iddx_full", "iddx_1", "iddx_2", "iddx_3", "iddx_4", "iddx_5",
    "mel_mitotic_index", "mel_thick_mm",
    "tbp_lv_dnn_lesion_confidence",

    # identifiers (not useful for model)
    "patient_id", "lesion_id"
]

train_df = train_df.drop(columns=[col for col in DROP_COLUMNS if col in train_df.columns])
val_df = val_df.drop(columns=[col for col in DROP_COLUMNS if col in val_df.columns])
holdout_df = holdout_df.drop(columns=[col for col in DROP_COLUMNS if col in holdout_df.columns])

### Separate Features and Targets

Target labels are separated before preprocessing.

This helps:
- Prevent accidental target modification
- Keep the preprocessing pipeline cleaner
- Maintain consistency during training

In [20]:
y_train = train_df["target"]
y_val = val_df["target"]
y_holdout = holdout_df["target"]

train_df = train_df.drop(columns=["target"])
val_df = val_df.drop(columns=["target"])
holdout_df = holdout_df.drop(columns=["target"])

In [21]:
train_df = train_df.drop(columns=["isic_id"])
val_df = val_df.drop(columns=["isic_id"])
holdout_df = holdout_df.drop(columns=["isic_id"])

### Feature Type Detection

Features are separated into:
- Numerical features
- Categorical features

Different preprocessing strategies will be applied to each type.

In [22]:
categorical_cols = train_df.select_dtypes(include=["object"]).columns.tolist()
numerical_cols = train_df.select_dtypes(exclude=["object"]).columns.tolist()

C:\Users\phang\AppData\Local\Temp\ipykernel_21248\2278989782.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = train_df.select_dtypes(include=["object"]).columns.tolist()


### Missing Value Handling

- Numerical features are imputed using the training median
- Categorical features are filled with `"unknown"`

Statistics are computed only from the training set to avoid leakage.

In [23]:
# Numerical
for col in numerical_cols:
    median = train_df[col].median()
    train_df[col] = train_df[col].fillna(median)
    val_df[col] = val_df[col].fillna(median)
    holdout_df[col] = holdout_df[col].fillna(median)

# Categorical
for col in categorical_cols:
    train_df[col] = train_df[col].fillna("unknown")
    val_df[col] = val_df[col].fillna("unknown")
    holdout_df[col] = holdout_df[col].fillna("unknown")

### Encoding Strategy

- Label Encoding is used for simplicity
- Encoders are fitted on training data only
- Same mapping is applied to validation and holdout sets

In [24]:
from sklearn.preprocessing import LabelEncoder
import numpy as np

encoders = {}

for col in categorical_cols:

    train_df[col] = train_df[col].astype(str)
    val_df[col] = val_df[col].astype(str)
    holdout_df[col] = holdout_df[col].astype(str)

    unique_values = train_df[col].unique().tolist()

    if "unknown" not in unique_values:
        unique_values.append("unknown")

    le = LabelEncoder()
    le.fit(unique_values)

    train_df[col] = train_df[col].apply(
        lambda x: x if x in le.classes_ else "unknown"
    )

    val_df[col] = val_df[col].apply(
        lambda x: x if x in le.classes_ else "unknown"
    )

    holdout_df[col] = holdout_df[col].apply(
        lambda x: x if x in le.classes_ else "unknown"
    )

    train_df[col] = le.transform(train_df[col])
    val_df[col] = le.transform(val_df[col])
    holdout_df[col] = le.transform(holdout_df[col])

    encoders[col] = le

### Numerical Feature Scaling

Numerical features are standardized using `StandardScaler`.

The scaler is fit only on the training data and then applied to validation and holdout sets.

In [25]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

train_df[numerical_cols] = scaler.fit_transform(train_df[numerical_cols])
val_df[numerical_cols] = scaler.transform(val_df[numerical_cols])
holdout_df[numerical_cols] = scaler.transform(holdout_df[numerical_cols])

In [26]:
train_processed = train_df.copy()
val_processed = val_df.copy()
holdout_processed = holdout_df.copy()

train_processed["target"] = y_train
val_processed["target"] = y_val
holdout_processed["target"] = y_holdout

train_processed["isic_id"] = train_ids
val_processed["isic_id"] = val_ids
holdout_processed["isic_id"] = holdout_ids

In [27]:
print("\n===== FINAL DATASET SHAPES =====")

print("Train:", train_processed.shape)
print("Validation:", val_processed.shape)
print("Holdout:", holdout_processed.shape)


===== FINAL DATASET SHAPES =====
Train: (320848, 44)
Validation: (38002, 44)
Holdout: (42209, 44)


In [28]:
print("\n===== FEATURE SUMMARY =====")

print("Numerical features:", len(numerical_cols))
print("Categorical features:", len(categorical_cols))
print("Total features:", len(numerical_cols) + len(categorical_cols))


===== FEATURE SUMMARY =====
Numerical features: 34
Categorical features: 8
Total features: 42


### Image Path Generation

Image file paths are generated for each sample.

These paths will later be used by:
- PyTorch datasets
- Image transforms
- Multimodal training pipelines

In [29]:
from pathlib import Path

IMAGE_DIR = Path("../data/raw/ISIC_2024_Training_Input")

train_processed["image_path"] = train_processed["isic_id"].apply(
    lambda x: str(IMAGE_DIR / f"{x}.jpg")
)

val_processed["image_path"] = val_processed["isic_id"].apply(
    lambda x: str(IMAGE_DIR / f"{x}.jpg")
)

holdout_processed["image_path"] = holdout_processed["isic_id"].apply(
    lambda x: str(IMAGE_DIR / f"{x}.jpg")
)

In [30]:
print("\n===== REMAINING MISSING VALUES =====")

print(
    train_processed.isnull().sum().sum()
)

print(
    val_processed.isnull().sum().sum()
)

print(
    holdout_processed.isnull().sum().sum()
)


===== REMAINING MISSING VALUES =====
0
0
0


In [31]:
from PIL import Image

bad_images = []

all_image_paths = pd.concat([
    train_processed["image_path"],
    val_processed["image_path"],
    holdout_processed["image_path"]
])

for path in all_image_paths:

    try:
        img = Image.open(path)
        img.verify()

    except:
        bad_images.append(path)

print("Corrupted images:", len(bad_images))

Corrupted images: 0


In [32]:
import joblib
import os

os.makedirs("../artifacts", exist_ok=True)

joblib.dump(
    scaler,
    "../artifacts/scaler.pkl"
)

joblib.dump(
    encoders,
    "../artifacts/label_encoders.pkl"
)

joblib.dump(
    numerical_cols,
    "../artifacts/numerical_cols.pkl"
)

joblib.dump(
    categorical_cols,
    "../artifacts/categorical_cols.pkl"
)

feature_columns = train_df.columns.tolist()

joblib.dump(
    feature_columns,
    "../artifacts/feature_columns.pkl"
)

print("Artifacts saved successfully!")

Artifacts saved successfully!


In [33]:
train_processed.to_csv("../data/processed/train_processed.csv", index=False)
val_processed.to_csv("../data/processed/val_processed.csv", index=False)
holdout_processed.to_csv("../data/processed/holdout_processed.csv", index=False)

print("Preprocessed data saved!")

Preprocessed data saved!


In [34]:
import sklearn
import pandas as pd

print("\n===== ENVIRONMENT INFO =====")

print("Pandas version:", pd.__version__)
print("Scikit-learn version:", sklearn.__version__)


===== ENVIRONMENT INFO =====
Pandas version: 3.0.2
Scikit-learn version: 1.8.0


## Preprocessing Complete

The processed datasets are now ready for:
- Exploratory Data Analysis (EDA)
- PyTorch Dataset creation
- Tabular model training
- Multimodal learning pipelines